## Altas interiores y exteriores 2020

En este notebook procesaremos un JSON que contiene, para cada municipio, las variaciones residenciales interiores(movimientos de población entre los diferentes municipios españoles) y exteriores(movimientos desde o hacia el extranjero).  
Obtendremos cuatro DataFrame con la siguiente información común:
- Y2020
- Nombre Provincia
- Codigo Provincia
- Nombre Municipio
- Codigo Municipio

Y en cada uno de de los DataFrame, uno de los siguientes campos:

- Altas Interiores
- Altas Exteriores
- Bajas Interiores
- Bajas Exteriores

In [1]:
import pandas as pd
import numpy as np
import requests

In [2]:
path = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/48507?tip=AM&'
json_request = requests.get(path).json()

In [3]:
MunAltaIntList = list()
MunAltaExtList = list()
AltaIntList = list()
AltaExtList = list()
MunBajaIntList = list()
MunBajaExtList = list()
BajaIntList = list()
BajaExtList = list()

In [4]:
for dato in json_request: 
    for i in range(0,len(dato['Data'])):
        if 'Altas Interiores, Total, Total' in dato['Nombre']:
            MunAltaIntList.append(dato['Nombre'])
            AltaIntList.append(dato['Data'][i]['Valor'])
        elif 'Altas Exteriores, Total, Total' in dato['Nombre']:
            MunAltaExtList.append(dato['Nombre'])
            AltaExtList.append(dato['Data'][i]['Valor']) 
        elif 'Bajas Interiores, Total, Total' in dato['Nombre']:
            MunBajaIntList.append(dato['Nombre'])
            BajaIntList.append(dato['Data'][i]['Valor']) 
        elif 'Bajas Exteriores, Total, Total' in dato['Nombre']:
            MunBajaExtList.append(dato['Nombre'])
            BajaExtList.append(dato['Data'][i]['Valor']) 

In [5]:
AltasInteriores20 = pd.DataFrame({
        'Municipios' : MunAltaIntList,
        'Altas Interiores': AltaIntList
    })
AltasExteriores20 = pd.DataFrame({
        'Municipios' : MunAltaExtList,
        'Altas Exteriores': AltaExtList
    })
BajasInteriores20 = pd.DataFrame({
        'Municipios' : MunBajaIntList,
        'Bajas Interiores': BajaIntList
    })
BajasExteriores20 = pd.DataFrame({
        'Municipios' : MunBajaExtList,
        'Bajas Exteriores': BajaExtList
    })

In [6]:
AltasInteriores20.head()

,Municipios,Altas Interiores
0,"Total, Altas Interiores, Total, Total",1519606.0
1,"44001-Ababuj, Altas Interiores, Total, Total",1.0
2,"40001-Abades, Altas Interiores, Total, Total",32.0
3,"10001-Abadía, Altas Interiores, Total, Total",32.0
4,"27001-Abadín, Altas Interiores, Total, Total",72.0


In [7]:
AltasExteriores20.head()

,Municipios,Altas Exteriores
0,"Total, Altas Exteriores, Total, Total",523618.0
1,"44001-Ababuj, Altas Exteriores, Total, Total",0.0
2,"40001-Abades, Altas Exteriores, Total, Total",0.0
3,"10001-Abadía, Altas Exteriores, Total, Total",0.0
4,"27001-Abadín, Altas Exteriores, Total, Total",7.0


In [8]:
BajasInteriores20.head()

,Municipios,Bajas Interiores
0,"Total, Bajas Interiores, Total, Total",1519606.0
1,"44001-Ababuj, Bajas Interiores, Total, Total",2.0
2,"40001-Abades, Bajas Interiores, Total, Total",23.0
3,"10001-Abadía, Bajas Interiores, Total, Total",4.0
4,"27001-Abadín, Bajas Interiores, Total, Total",48.0


In [9]:
BajasExteriores20.head()

,Municipios,Bajas Exteriores
0,"Total, Bajas Exteriores, Total, Total",271225.0
1,"44001-Ababuj, Bajas Exteriores, Total, Total",0.0
2,"40001-Abades, Bajas Exteriores, Total, Total",1.0
3,"10001-Abadía, Bajas Exteriores, Total, Total",0.0
4,"27001-Abadín, Bajas Exteriores, Total, Total",0.0


Eliminamos la primera fila(el total) de cada uno de los DataFrame:

In [10]:
AltasInteriores20.drop(0, inplace = True)
AltasExteriores20.drop(0, inplace = True)
BajasInteriores20.drop(0, inplace = True)
BajasExteriores20.drop(0, inplace = True)

Cargamos la tabla 01_Output_ProvMun_20.xlsx para posteriormente realizar el merge con las tablas anteriores:

In [11]:
ProvMun = pd.read_excel('01_Output_ProvMun_20.xlsx', dtype='object') 
ProvMun.head()

,Nombre Provincia,Codigo Provincia,Nombre Municipio,Codigo Municipio
0,Araba/Álava,01,Agurain/Salvatierra,01051
1,Araba/Álava,01,Alegría-Dulantzi,01001
2,Araba/Álava,01,Amurrio,01002
3,Araba/Álava,01,Añana,01049
4,Araba/Álava,01,Aramaio,01003


Añadimos el campo Código Municipio a los DataFrame asignando el valor de cada fila en función del código que contiene el campo Municipio:

In [12]:
MunList = ProvMun['Codigo Municipio'].tolist()

In [13]:
def check_list(x):
    for l in MunList:
        if l in x:
            return l
    return ''

In [14]:
AltasInteriores20['Codigo Municipio'] = AltasInteriores20['Municipios'].map(lambda x: check_list(x))
AltasExteriores20['Codigo Municipio'] = AltasExteriores20['Municipios'].map(lambda x: check_list(x))
BajasInteriores20['Codigo Municipio'] = BajasInteriores20['Municipios'].map(lambda x: check_list(x))
BajasExteriores20['Codigo Municipio'] = BajasExteriores20['Municipios'].map(lambda x: check_list(x))

In [15]:
AltasInteriores20 = AltasInteriores20.merge(ProvMun, on='Codigo Municipio')
AltasExteriores20 = AltasExteriores20.merge(ProvMun, on='Codigo Municipio')
BajasInteriores20 = BajasInteriores20.merge(ProvMun, on='Codigo Municipio')
BajasExteriores20 = BajasExteriores20.merge(ProvMun, on='Codigo Municipio')

In [16]:
AltasInteriores20['Y2020'] = 2020
ColNue = ['Y2020','Nombre Provincia', 'Codigo Provincia', 'Nombre Municipio','Codigo Municipio', 'Altas Interiores']
AltasInteriores20 = AltasInteriores20[ColNue]
AltasInteriores20.head()

,Y2020,Nombre Provincia,Codigo Provincia,Nombre Municipio,Codigo Municipio,Altas Interiores
0,2020,Teruel,44,Ababuj,44001,1.0
1,2020,Segovia,40,Abades,40001,32.0
2,2020,Cáceres,10,Abadía,10001,32.0
3,2020,Lugo,27,Abadín,27001,72.0
4,2020,Bizkaia,48,Abadiño,48001,213.0


In [17]:
AltasExteriores20['Y2020'] = 2020
ColNue = ['Y2020','Nombre Provincia', 'Codigo Provincia', 'Nombre Municipio','Codigo Municipio', 'Altas Exteriores']
AltasExteriores20 = AltasExteriores20[ColNue]
AltasExteriores20.head()

,Y2020,Nombre Provincia,Codigo Provincia,Nombre Municipio,Codigo Municipio,Altas Exteriores
0,2020,Teruel,44,Ababuj,44001,0.0
1,2020,Segovia,40,Abades,40001,0.0
2,2020,Cáceres,10,Abadía,10001,0.0
3,2020,Lugo,27,Abadín,27001,7.0
4,2020,Bizkaia,48,Abadiño,48001,25.0


In [18]:
BajasInteriores20['Y2020'] = 2020
ColNue = ['Y2020','Nombre Provincia', 'Codigo Provincia', 'Nombre Municipio','Codigo Municipio', 'Bajas Interiores']
BajasInteriores20 = BajasInteriores20[ColNue]
BajasInteriores20.head()

,Y2020,Nombre Provincia,Codigo Provincia,Nombre Municipio,Codigo Municipio,Bajas Interiores
0,2020,Teruel,44,Ababuj,44001,2.0
1,2020,Segovia,40,Abades,40001,23.0
2,2020,Cáceres,10,Abadía,10001,4.0
3,2020,Lugo,27,Abadín,27001,48.0
4,2020,Bizkaia,48,Abadiño,48001,250.0


In [19]:
BajasExteriores20['Y2020'] = 2020
ColNue = ['Y2020','Nombre Provincia', 'Codigo Provincia', 'Nombre Municipio','Codigo Municipio', 'Bajas Exteriores']
BajasExteriores20 = BajasExteriores20[ColNue]
BajasExteriores20.head()

,Y2020,Nombre Provincia,Codigo Provincia,Nombre Municipio,Codigo Municipio,Bajas Exteriores
0,2020,Teruel,44,Ababuj,44001,0.0
1,2020,Segovia,40,Abades,40001,1.0
2,2020,Cáceres,10,Abadía,10001,0.0
3,2020,Lugo,27,Abadín,27001,0.0
4,2020,Bizkaia,48,Abadiño,48001,17.0


In [20]:
AltasInteriores20.to_excel('06_Output_Altas_Interiores_20.xlsx', header = True, index = False)
AltasExteriores20.to_excel('06_Output_Altas_Exteriores_20.xlsx', header = True, index = False)
BajasInteriores20.to_excel('06_Output_Bajas_Interiores_20.xlsx', header = True, index = False)
BajasExteriores20.to_excel('06_Output_Bajas_Exteriores_20.xlsx', header = True, index = False)